In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval
except Exception:
    pass

# This guide is about binning; silence the unrelated level-rename deprecation so
# the output stays on topic.
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
from dataclasses import dataclass

import numpy as np

from dataeval import Metadata
from dataeval.protocols import DatasetMetadata, DatumMetadata

In [3]:
@dataclass
class BoxTarget:
    """A minimal object detection target: boxes, labels, and scores."""

    boxes: np.ndarray
    labels: np.ndarray
    scores: np.ndarray


class CrowdingDataset:
    """A synthetic detection dataset whose crowding correlates with altitude."""

    def __init__(self, detections_per_image: list[int]) -> None:
        rng = np.random.default_rng(0)
        self._counts = detections_per_image
        # Precomputed so that every read of an item returns the same values.
        # Sorted, so altitude rises with the image index and therefore with crowding.
        self._altitudes = np.sort(rng.uniform(50.0, 400.0, len(detections_per_image)))
        self._areas = [rng.uniform(20.0, 200.0, count) for count in detections_per_image]
        self.metadata = DatasetMetadata(id="crowding-demo", index2label={0: "person", 1: "car"})

    def __len__(self) -> int:
        return len(self._counts)

    def __getitem__(self, index: int) -> tuple[np.ndarray, BoxTarget, DatumMetadata]:
        count = self._counts[index]
        target = BoxTarget(
            boxes=np.tile(np.array([[0.0, 0.0, 10.0, 10.0]]), (count, 1)),
            labels=np.arange(count) % 2,
            scores=np.ones(count),
        )
        # Weather is balanced across images - half clear, half rainy - but correlated
        # with crowding: three quarters of the sparse images are clear and three
        # quarters of the crowded ones are rainy.
        if index < len(self._counts) // 2:
            weather = "rainy" if index % 4 == 0 else "clear"
        else:
            weather = "clear" if index % 4 == 0 else "rainy"
        # altitude_m and weather describe the image; box_area describes each box.
        datum_metadata: DatumMetadata = DatumMetadata(
            id=index,
            **{
                "altitude_m": float(self._altitudes[index]),
                "weather": weather,
                "box_area": self._areas[index].tolist(),
            },
        )
        return np.zeros((3, 32, 32), dtype=np.uint8), target, datum_metadata


counts = [1] * 20 + [8] * 20
dataset = CrowdingDataset(counts)
metadata = Metadata(dataset, auto_bin_method="uniform_count", exclude=["id"])

print(f"images:     {metadata.level_counts['image']}")
print(f"detections: {metadata.level_counts['instance']}")

images:     40
detections: 180


In [4]:
for name, info in metadata.factor_info.items():
    print(f"{name:12s} level={info.level:9s} type={info.factor_type}")

altitude_m   level=image     type=continuous
box_area     level=instance  type=continuous
weather      level=image     type=categorical


In [5]:
def companion(md: Metadata, name: str) -> str:
    """Name of the column holding a factor's discretized values."""
    info = md.factor_info[name]
    if info.is_binned:
        return f"{name}↕"
    return f"{name}#" if info.is_digitized else name


altitude_bins = companion(metadata, "altitude_m")
print(f"altitude_m is stored discretized in {altitude_bins!r}")

altitude_m is stored discretized in 'altitude_m↕'


In [6]:
altitudes = metadata.rows_at("image")["altitude_m"].to_numpy()
replicated = np.repeat(altitudes, counts)

print(f"quartile edges over the 40 images:     {np.round(np.percentile(altitudes, [0, 25, 50, 75, 100]), 1)}")
print(f"quartile edges over the 180 detections: {np.round(np.percentile(replicated, [0, 25, 50, 75, 100]), 1)}")

quartile edges over the 40 images:     [ 51.  157.7 263.9 312.7 399. ]
quartile edges over the 180 detections: [ 51.  277.7 305.3 361.3 399. ]


In [7]:
at_image = metadata.rows_at("image")[altitude_bins].to_list()
detections = metadata.rows_at("instance")
gathered = [detections.filter(detections["item_index"] == i)[altitude_bins][0] for i in range(len(counts))]

print(f"identical read from either level: {at_image == gathered}")

identical read from either level: True


In [8]:
sparse = CrowdingDataset([2, 1, 2, 0])
sparse_metadata = Metadata(sparse, exclude=["id"])
sparse_bins = companion(sparse_metadata, "altitude_m")

print(f"images:     {sparse_metadata.level_counts['image']}")
print(f"detections: {sparse_metadata.level_counts['instance']}")
print(f"altitude bins at image level: {sparse_metadata.rows_at('image')[sparse_bins].to_list()}")

images:     4
detections: 5
altitude bins at image level: [0, 1, 2, 3]


In [9]:
image_rows = metadata.rows_at("image")["weather"].to_list()
detection_rows = metadata.rows_at("instance")["weather"].to_list()

print(f"weather over images:     clear={image_rows.count('clear')}, rainy={image_rows.count('rainy')}")
print(f"weather over detections: clear={detection_rows.count('clear')}, rainy={detection_rows.count('rainy')}")

weather over images:     clear=20, rainy=20
weather over detections: clear=55, rainy=125
